This notebook is based closely on demo_rdm_visualisation_92images.ipynb from the RSA toolbox.

# Visualising RDMs with the 92 images 

This notebook demonstrates how `rsatoolbox.vis.show_rdm` can be used to visualise RDMs. By setting the RDM's pattern_descriptors to `rsatoolbox.vis.Icon` instances, it is possible to automatically generate stacked image labels and other custom visualisations.

In [ ]:
# imports
import matplotlib.pyplot as plt
import matplotlib
from rsatoolbox import vis
from rsatoolbox import rdm
import numpy as np
import os
import scipy.io
from collections import defaultdict

%matplotlib inline 
# without this line, it might be necessary to call plt.show() for each figure

## Loading data and building RDMs instance with Icon pattern_descriptors
Before we begin, we need to load up the fMRI data from the 92 images dataset ([Kriegeskorte et al., 2008](https://www.sciencedirect.com/science/article/pii/S0896627308009434)).

In [ ]:
# supporting functions and constants
DEMO_DIR = os.getcwd()
NEURON_DIR = os.path.join(DEMO_DIR, "rsatoolbox_demos_2025/92imageData")


def neuron_2008_icons(**kwarg):
    """ Load Krigeskorte et al. (2008, Neuron) images as Icon instances."""
    mat_path = os.path.join(NEURON_DIR, "Kriegeskorte_Neuron2008_supplementalData.mat")
    mat = scipy.io.loadmat(mat_path)
    colors = plt.get_cmap('Accent', lut=16).colors
    markers = list(matplotlib.markers.MarkerStyle('').markers.keys())
    icons = defaultdict(list)
    for this_struct in mat["stimuli_92objs"][0]:
        # we're going to treat the 4 binary indicators (human, face, animal, natural) as a
        # base2 binary string, and index into this array of colors accordingly.
        index = int("".join(
                         [
                            str(this_struct[this_key][0,0]) for this_key in (
                             'human', 'face', 'animal', 'natural')
                         ]
                           ),
                    base=2)
        this_color = colors[index]
        icons['image'].append(vis.Icon(image = this_struct['image'],
                                       color = this_color,
                                    circ_cut = 'cut',
                                 border_type = 'conv',
                                border_width = 5,
                                       **kwarg))
        icons['string'].append(vis.Icon(string = this_struct['category'][0],
                                         color = this_color,
                                    font_color = this_color,
                                        **kwarg))
        icons['marker'].append(vis.Icon(marker = markers[index],
                                         color = this_color,
                                       **kwarg))
    return icons


def neuron_2008_rdms_fmri(**kwarg):
    """ Load Kriegeskorte et al. (2008, Neuron) fMRI RDMs as RDMs instance.
    All key-word arguments are passed to neuron_2008_icons."""
    mat_path = os.path.join(NEURON_DIR, "92_brainRDMs.mat")
    mat = scipy.io.loadmat(mat_path)
    icons = neuron_2008_icons(**kwarg)
    # insert leading dim to conform with rsatoolbox nrdm x ncon x ncon convention
    return rdm.concat(
        [
            rdm.RDMs(
                dissimilarities = this_rdm["RDM"][None, :, :],
                dissimilarity_measure = "1-rho",
                rdm_descriptors = dict(
                    zip(["ROI", "subject", "session"], this_rdm["name"][0].split(" | ")),
                    name = this_rdm["name"][0]
                ),
                pattern_descriptors = icons
            )
            for this_rdm in mat["RDMs"].flatten()
        ]
    )

In [ ]:
# as an example, load the Kriegeskorte et al., 2008 fMRI RDMs
rdms = neuron_2008_rdms_fmri()

The above code loaded an RDMs object that contains eight 92 x 92 RDMs (two sessions from four subjects). The RDMs contain lists of rsatoolbox.vis.icon.Icon objects as the pattern descriptors. 

## Plotting RDMs with text labels

To keep it simple, let's start with plotting the first 12 conditions: 

In [ ]:
rdms_small = rdms.subset_pattern('index', np.arange(12))
rdms_small = rdm.rank_transform(rdms_small)

In [ ]:
vis.show_rdm(rdms_small, show_colorbar = 'figure', cmap = 'classic', rdm_descriptor = 'name', figsize = (10,10),
             pattern_descriptor = 'index');

## Plotting with image labels
It might help to put some image labels on...

*(( There currently seems to be a bug where the icons are added to the bottom of the top RDM, rather than the bottom of the bottom RDM? Also, setting show_colorbar = 'figure' removes icons from the adjacent axis? Therefore, from now on, I'll only plot one RDM, with show_colorbar = 'panel'... ))*

In [ ]:
vis.show_rdm(rdms_small[0], show_colorbar = 'panel', cmap = 'classic', rdm_descriptor = 'name', figsize = (6,6),
             pattern_descriptor = 'image');

Notice how the image labels are scaled to avoid excessive overlap. However, they are rather small. Grouping them might help us get away with larger images. We will also tweak `icon_spacing`.

In [ ]:
vis.show_rdm(rdms_small[0], show_colorbar = 'panel', cmap = 'classic', rdm_descriptor = 'name', figsize = (6,6),
             pattern_descriptor = 'image', num_pattern_groups = 3, icon_spacing = 1.2);

Now the images are very clear (maybe too big in fact, unless you are preparing a figure for a slideshow). Notice how `show_rdm` adds white gridlines to mark the label groups. If you like, you can control this behavior with the `gridlines` argument.

For now though, let's scale up to the full insanity of 92 images...

In [ ]:
fig, ax, ret_val = vis.show_rdm(rdms[0], show_colorbar='panel', cmap='classic', rdm_descriptor='name', 
                                pattern_descriptor='image', num_pattern_groups=6, icon_spacing=.9,
             vmin=0., figsize=(10,10)
            );

Not too bad, considering how many images we are plotting. With this full view you can also see two other features that help guide the eye in large RDM visualisations:
* A carefully selected `num_pattern_groups` parameter really helps with picking out interesting conditions (notice how the faces are in grids 3-4 and 7-8 above).
* The `color` parameter that was used when instancing the Icons controls the color of the supporting lines and the circular outlines. This can also be used to convey category membership.

## Optimising image label display and saving figure
In general, you can achieve larger, clearer image labels by:
* plotting fewer RDMs in the same figure
* increasing `num_pattern_groups`
* decreasing `icon_spacing`
* adjusting `figsize`
* increasing `dpi` when saving, see below

In [ ]:
# to save the figure, remember to set bbox_inches='tight'.
# (Otherwise the image labels may get cropped out)
fig.savefig('temp_rdm.png', bbox_inches='tight', dpi=300)

The saved RDM image is plotted below. It may look crisper than the inline view we generated previously.
![rdm image](temp_rdm.png)

## Plot MDS

Here we make a scatterplot of the RDM reduced to two dimensions using MultiDimensional Scaling

In [ ]:
fig=vis.show_MDS(
    rdms[0,1],
    rdm_descriptor='name',
    pattern_descriptor='image', 
    icon_size=0.2, 
);
fig.set_size_inches(10,10)

There are also functions for other dimension reduction techniques, e.g `.show_iso` and `.show_tSNE`.
And the pattern descriptor could be changed from 'image' to 'string'. E.g.:

In [ ]:
fig=vis.show_iso(
    rdms[0],
    rdm_descriptor='name',
    pattern_descriptor='string', 
);
fig.set_size_inches(10,10)